# Checking the Equivalence of Regular Expressions

In order to check whether two regular expressions $r_1$ and $r_2$ are *equivalent*, perform the 
following steps:
- convert the regular expressions $r_1$ and $r_2$ into *NFAs*
  $F_1$ and $F_2$ such that $L(r_1) = L(F_1)$ and $L(r_2) = L(F_2)$,
- convert the *NFAs* $F_1$ and $F_2$ into *DFAs*
  $D_1$ and $D_2$ such that $L(D_1) = L(F_1)$ and $L(D_2) = L(F_2)$
- check whether both $L(D_1) \backslash L(D_2)$ and $L(D_2) \backslash L(D_1)$ are empty.

In [ ]:
#load "AutomataTypes.fsx"
#load "RegExpToNfa.fsx"
#load "NfaToDfa.fsx"
#load "Minimize.fsx"

In [ ]:
open AutomataTypes
open RegExpToNfa
open NfaToDfa
open Minimize

`regexp2nfa` and `nfa2dfa` are exported from their teaching notebooks. `cartesianProduct` and reachability are reused from the minimization notebook. F# checks these generic interfaces across notebook boundaries without type-checker suppressions.

Given two sets `A` and `B`, the function `cartesianProduct A B` computes the 
<em style="color:blue">cartesian product</em> $A \times B$ which is defined as
$$ A \times B := \{ (x, y) \mid x \in A \wedge y \in B \}. $$

In [ ]:
cartesianProduct (set [1;2]) (set ['a';'b'])

Given to deterministic *FSMs* `F1` and `F2`, the expression `fsmComplement first second` computes a deterministic 
*FSM* that recognizes the language  $L(F_1)\backslash L(F_2)$.

This function is language **difference**, rather than unary complement. Product states may pair two different state types. Both DFAs must be complete over the same alphabet; that precondition is essential for the second automaton's non-accepting states to represent its complement.

In [ ]:
let fsmComplement (first: Dfa<'first>) (second: Dfa<'second>) : Dfa<'first * 'second> =
    requireComplete first
    requireComplete second
    if first.Alphabet <> second.Alphabet then invalidArg "second" "Language difference requires the same alphabet."
    let states = cartesianProduct first.States second.States
    let transitions =
        seq { for q1, q2 in states do
                  for c in first.Alphabet do
                      yield ((q1, q2), c), (first.Transitions[q1,c], second.Transitions[q2,c]) }
        |> Map.ofSeq
    { States = states; Alphabet = first.Alphabet; Transitions = transitions
      Start = first.Start, second.Start
      Accepting = cartesianProduct first.Accepting (Set.difference second.States second.Accepting) }

Given a regular expression $r$ and an alphabet $\Sigma$, the function $\texttt{regexp2DFA}(r, \Sigma)$
computes a *DFA* that accepts the language specified by $r$.

In [ ]:
let regexp2DFA expression alphabet : Dfa<Set<int>> =
    regexp2nfa alphabet expression |> nfa2dfa

Given a deterministic *FSM* $F$ the function 
`isEmpty dfa` checks whether the language accepted by $F$ is empty.
In this function, the variable `Reachable` is the set of those states that are already known to be reachable
from the start state `q0`. `NewFound` are those states that can be reached from a state in the set 
`Reachable`.  When we find no new states that are reachable, the iteration stops and we check whether
there is a state that is both reachable and acceptable because in that case the language is not empty.

In [ ]:
let isEmpty (dfa: Dfa<'state>) : bool =
    requireComplete dfa
    let reached = reachable dfa.Start dfa.Alphabet dfa.Transitions
    Set.isEmpty (Set.intersect reached dfa.Accepting)

The function `regExpEquiv` takes three arguments:
- $r_1$ and $r_2$ are regular expressions,
- $\Sigma$ is the alphabet used in these regular expressions.

The function returns `true` iff $r_1 \doteq r_2$, i.e. if $r_1$ and $r_2$ are equivalent.

In [ ]:
let regExpEquiv first second alphabet : bool =
    let firstDfa = regexp2DFA first alphabet
    let secondDfa = regexp2DFA second alphabet
    isEmpty (fsmComplement firstDfa secondDfa) && isEmpty (fsmComplement secondDfa firstDfa)

The notebook `10-Test-Equivalence.ipynb` can be used to test this function.